# Laya: zero-shot baseline and supervised fine-tuning

**Default:** binary detection pilot, 10,000 training edges, 1,000 checkpoint-selection edges, 2,000 calibration edges, and 2,000 test edges. Training sampling balances binary classes; evaluation sampling preserves their proportions. Set TASK_MODE to "all" for detection plus all training-set tactics and techniques. Use a new RUN_NAME when changing an experiment.

**Before running:** edit the three paths and INPUT_FORMAT in section 2. Use parquet files saved before log transformation/scaling for INPUT_FORMAT="raw". If saved after scaling, choose "log1p_scaled" and provide preprocessing_objects.joblib. A CUDA GPU is recommended. Model weights download on first use; this notebook does not upload data or models.


## 1. Install and import

Use your existing CUDA-enabled PyTorch environment. This cell installs the Laya version used in your experiment and data dependencies. Restart the kernel if pip replaces an already-imported package.


In [1]:
%pip install "laya==0.4.15" "pandas>=2.0" pyarrow scikit-learn scipy joblib tqdm matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import ast
import copy
import gc
import hashlib
import importlib.metadata
import json
import random
import time
import warnings
from pathlib import Path
from dataclasses import asdict

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.special import expit
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score,
    f1_score, accuracy_score, brier_score_loss, precision_recall_curve,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from laya.train import (
    TrainConfig, load_checkpoint, save_checkpoint, train_model,
    items_from_rows, calibration_records, to_internal,
)
from laya.common import build_head, clamp_temperature, QTYPES, temp_bucket

assert importlib.metadata.version("laya") == "0.4.15", "Restart after installing laya 0.4.15."
print("PyTorch:", torch.__version__, "| Laya:", importlib.metadata.version("laya"))


PyTorch: 2.5.1+cu121 | Laya: 0.4.15


## 2. Experiment settings

Choose "binary" for the first run. Choose "all" for independent binary decisions for every tactic and technique. All tasks use the same model; no technique predictions are gated by attack predictions.

INPUT_FORMAT supports "raw", "log1p", and "log1p_scaled", matching your GraphSAGE preparation. Do not guess based only on the column names: they stay the same after transformation.

For the final full evaluation, set TEST_MAX_ROWS=None **before the planned final test**. Avoid repeatedly changing settings in response to test results. Keep all sampled IDs and run metadata for reproducibility.


In [3]:
DATA_DIR = Path("processed_edges")
TRAIN_PATH = DATA_DIR / "train_edges.parquet"
VALIDATION_PATH = DATA_DIR / "validation_edges.parquet"
TEST_PATH = DATA_DIR / "test_edges.parquet"
INPUT_FORMAT = "raw"  # raw | log1p | log1p_scaled
PREPROCESSING_PATH = Path("preprocessing_objects.joblib")

TASK_MODE = "binary"  # binary | all
INCLUDE_NETWORK_CONTEXT = True  # computed within each full five-minute window
RUN_NAME = "laya_binary_pilot_v1"
OUTPUT_DIR = Path("laya_runs") / RUN_NAME
ALLOW_OVERWRITE = False
SEED = 42

TRAIN_MAX_ROWS = 10_000
TRAIN_ATTACK_FRACTION = 0.50  # training only; None preserves training prevalence
SELECT_MAX_ROWS = 1_000
CALIBRATION_MAX_ROWS = 2_000
TEST_MAX_ROWS = 2_000
AUX_NEGATIVES_PER_POSITIVE = 3  # training only, per tactic/technique; None keeps all
MIN_THRESHOLD_POSITIVES = 10
MIN_THRESHOLD_NEGATIVES = 10

MODEL_ID = "convaiinnovations/laya"
MODEL_REVISION = "main"  # resolved snapshot commit is recorded; pin it for repeat experiments
MAX_LEN = 512
HEAD_MAX_LEN = 192
EPOCHS = 4
MICRO_BATCH = 2
GRAD_ACCUM = 32  # effective batch 64 decisions
EVAL_BATCH_SIZE = 4
ENCODER_LR = 2.5e-5
HEAD_LR = 1e-4
FREEZE_ENCODER = False
LOSS = "soft-ce"  # ordinary supervised CE for one-hot targets; optional: rlcd
ALLOW_CPU_TRAINING = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU detected. Baseline inference works; full fine-tuning will be slow.")
assert TASK_MODE in {"binary", "all"}
assert INPUT_FORMAT in {"raw", "log1p", "log1p_scaled"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUTPUT_DIR / "run_config.json").exists() and not ALLOW_OVERWRITE:
    raise FileExistsError("This run already exists. Choose a new RUN_NAME, or intentionally set ALLOW_OVERWRITE=True.")

RUN_CONFIG = {k: str(v) if isinstance(v, (Path, torch.device)) else v
              for k, v in list(globals().items())
              if k.isupper() and isinstance(v, (str, int, float, bool, type(None), Path, torch.device))}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2))
(OUTPUT_DIR / "versions.json").write_text(json.dumps({
    p: importlib.metadata.version(p)
    for p in ["laya", "torch", "transformers", "pandas", "numpy", "scikit-learn"]
}, indent=2))


GPU: NVIDIA RTX 500 Ada Generation Laptop GPU


145

## 3. Load, restore raw units, and validate the splits

The model receives a feature whitelist only. Labels, annotation flags, tactic/technique counts, source period, dates, and literal IP addresses never enter its text.

Missing attack annotations are unknown, not negative labels. For annotated attack rows, unlisted labels are treated as absent, matching a closed-label-set benchmark. If those annotations are incomplete, row-level flags cannot solve that limitation; label-level observation masks would be needed. Benign rows are negatives for every tactic and technique.


In [4]:
NUMERIC_FEATURES = [
    "connection_count", "total_duration", "average_duration", "maximum_duration",
    "total_orig_bytes", "total_resp_bytes", "total_orig_packets", "total_resp_packets",
    "total_orig_ip_bytes", "total_resp_ip_bytes", "total_missed_bytes",
    "unique_destination_ports", "most_common_destination_port", "tcp_count", "udp_count", "icmp_count",
]
LOG_FEATURES = [c for c in NUMERIC_FEATURES if c != "most_common_destination_port"]
INTEGER_FEATURES = [c for c in NUMERIC_FEATURES if "duration" not in c]
KEY_COLUMNS = ["window_start", "src_ip_zeek", "dest_ip_zeek"]

def label_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, np.ndarray, set)):
        return sorted(set(str(x).strip() for x in value if x is not None and str(x).strip()))
    if isinstance(value, str):
        value = value.strip()
        if not value or value.lower() in {"none", "nan", "null"}:
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            raise ValueError(f"Expected a label list, got {value!r}")
        return label_list(parsed)
    if pd.isna(value):
        return []
    raise TypeError(f"Unsupported label container: {type(value)}")

def parse_flag(value):
    if value is None or pd.isna(value):
        raise ValueError("Annotation flags must not be missing.")
    if isinstance(value, str):
        if value.lower() in {"true", "1"}: return True
        if value.lower() in {"false", "0"}: return False
        raise ValueError(f"Unrecognized annotation flag: {value}")
    if value not in (0, 1, False, True):
        raise ValueError(f"Unrecognized annotation flag: {value}")
    return bool(value)

def restore_raw_features(df, input_format, preprocessing_path=None):
    df = df.copy()
    df[NUMERIC_FEATURES] = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="raise").astype(float)
    if input_format == "log1p_scaled":
        objects = joblib.load(preprocessing_path)  # use only your own trusted joblib file
        scaler = objects["edge_scaler"]
        order = list(getattr(scaler, "feature_names_in_", NUMERIC_FEATURES))
        if order != NUMERIC_FEATURES:
            raise ValueError("Scaler feature order differs from the supplied GraphSAGE notebook.")
        df[NUMERIC_FEATURES] = scaler.inverse_transform(df[NUMERIC_FEATURES])
    if input_format in {"log1p", "log1p_scaled"}:
        df[LOG_FEATURES] = np.expm1(df[LOG_FEATURES])
    a = df[NUMERIC_FEATURES].to_numpy()
    if not np.isfinite(a).all() or (a < -1e-5).any():
        raise ValueError("Features must be finite and nonnegative in raw units. Check INPUT_FORMAT.")
    df[NUMERIC_FEATURES] = df[NUMERIC_FEATURES].clip(lower=0)
    for col in INTEGER_FEATURES:
        a = df[col].to_numpy()
        if not np.allclose(a, np.rint(a), rtol=1e-5, atol=1e-3):
            raise ValueError(f"{col} is not integer-like. The file may contain transformed features.")
        df[col] = np.rint(a)
    if (df.connection_count < 1).any():
        raise ValueError("connection_count must be at least 1 in raw units.")
    if (df.average_duration > df.maximum_duration + 1e-4).any():
        raise ValueError("Average duration exceeds maximum duration. Check feature conversion.")
    if not np.allclose(df.total_duration, df.average_duration * df.connection_count, rtol=2e-4, atol=2e-3):
        raise ValueError("Duration totals and averages are inconsistent. Check INPUT_FORMAT.")
    return df

def load_split(path, name):
    if not path.is_file():
        raise FileNotFoundError(f"Edit section 2: {path.resolve()} does not exist.")
    df = pd.read_parquet(path).reset_index(drop=True)
    required = set(KEY_COLUMNS + NUMERIC_FEATURES + [
        "most_common_service", "most_common_connection_state", "binary_target",
        "tactic_targets", "mitre_targets",
    ])
    if required - set(df.columns):
        raise ValueError(f"{name}: missing columns {sorted(required - set(df.columns))}")
    df = restore_raw_features(df, INPUT_FORMAT, PREPROCESSING_PATH)
    df["window_start"] = pd.to_datetime(df["window_start"], errors="raise")
    if df[KEY_COLUMNS].isna().any().any() or df.duplicated(KEY_COLUMNS).any():
        raise ValueError(f"{name}: missing or duplicate edge identifiers.")
    if not df.binary_target.isin([0, 1]).all():
        raise ValueError(f"{name}: binary_target must contain only 0 and 1.")
    df["binary_target"] = df.binary_target.astype(int)
    for family in ["tactic", "mitre"]:
        col, flag = family + "_targets", "has_" + family + "_target"
        df[col] = df[col].map(label_list)
        nonempty = df[col].map(bool)
        if flag not in df:
            df[flag] = nonempty
        else:
            df[flag] = df[flag].map(parse_flag)
            if not (df[flag] == nonempty).all():
                raise ValueError(f"{name}: {flag} conflicts with {col}.")
        if ((df.binary_target == 0) & nonempty).any():
            raise ValueError(f"{name}: benign rows contain attack labels.")
        df["observed_" + family] = (df.binary_target == 0) | df[flag]
    for col in ["most_common_service", "most_common_connection_state"]:
        df[col] = df[col].fillna("unknown").astype(str)
    df["edge_id"] = [hashlib.sha256(json.dumps([str(x) for x in key]).encode()).hexdigest()[:24]
                     for key in df[KEY_COLUMNS].itertuples(index=False, name=None)]
    df["split"] = name
    return df

train_full = load_split(TRAIN_PATH, "train")
validation_full = load_split(VALIDATION_PATH, "validation")
test_full = load_split(TEST_PATH, "test")
for a, b in [(train_full, validation_full), (train_full, test_full), (validation_full, test_full)]:
    if set(a.window_start) & set(b.window_start):
        raise ValueError("A five-minute window occurs in multiple splits. Fix the input split files.")
display(pd.DataFrame([
    {"split": n, "rows": len(d), "windows": d.window_start.nunique(),
     "attack_fraction": d.binary_target.mean(),
     "attack_missing_tactic": int(((d.binary_target == 1) & ~d.has_tactic_target).sum()),
     "attack_missing_technique": int(((d.binary_target == 1) & ~d.has_mitre_target).sum())}
    for n, d in [("train", train_full), ("validation", validation_full), ("test", test_full)]
]))


,split,rows,windows,attack_fraction,attack_missing_tactic,attack_missing_technique
0,train,541763,20149,0.080048,0,68
1,validation,67130,2519,0.083614,0,7
2,test,69161,2518,0.081983,0,265


## 4. Network context and development subsets

Context is computed from the complete input split before sampling, separately within each window. Only traffic observations are aggregated. This is an end-of-window classifier, not a detector that sees future traffic before it arrives.

Validation windows are divided into checkpoint-selection and calibration/threshold subsets. No validation or test labels are used for gradient updates. Evaluation subsets are sampled proportionally by binary class, not balanced. Rare-label support may be small in a pilot and is reported explicitly.


In [5]:
def add_network_context(df):
    df = df.copy()
    out = df.groupby(["window_start", "src_ip_zeek"], sort=False)
    inc = df.groupby(["window_start", "dest_ip_zeek"], sort=False)
    df["src_unique_destinations"] = out.dest_ip_zeek.transform("nunique")
    df["src_connection_count"] = out.connection_count.transform("sum")
    df["src_sent_bytes"] = out.total_orig_bytes.transform("sum")
    df["dst_unique_sources"] = inc.src_ip_zeek.transform("nunique")
    df["dst_connection_count"] = inc.connection_count.transform("sum")
    df["dst_response_bytes"] = inc.total_resp_bytes.transform("sum")
    return df

if INCLUDE_NETWORK_CONTEXT:
    train_full = add_network_context(train_full)
    validation_full = add_network_context(validation_full)
    test_full = add_network_context(test_full)

def sample_rows(df, limit, seed, attack_fraction=None):
    if limit is None or limit >= len(df):
        return df.copy().reset_index(drop=True)
    if limit < 2:
        raise ValueError("Sample size must be at least 2.")
    if attack_fraction is None:
        if df.binary_target.nunique() == 2 and df.binary_target.value_counts().min() >= 2:
            selected, _ = train_test_split(np.arange(len(df)), train_size=limit,
                                           stratify=df.binary_target, random_state=seed)
            return df.iloc[selected].reset_index(drop=True)
        return df.sample(n=limit, random_state=seed).reset_index(drop=True)
    n_pos = min(int(round(limit * attack_fraction)), int(df.binary_target.sum()))
    n_neg = min(limit - n_pos, int((df.binary_target == 0).sum()))
    n_pos = min(limit - n_neg, int(df.binary_target.sum()))
    sampled = pd.concat([
        df[df.binary_target == 1].sample(n=n_pos, random_state=seed),
        df[df.binary_target == 0].sample(n=n_neg, random_state=seed),
    ]).sample(frac=1, random_state=seed)
    return sampled.reset_index(drop=True)

window_labels = validation_full.groupby("window_start").binary_target.max()
if window_labels.value_counts().min() < 2 or window_labels.nunique() < 2:
    raise ValueError("Need benign and attack validation windows for both development subsets.")
select_windows, calibration_windows = train_test_split(
    window_labels.index.to_numpy(), test_size=0.5, stratify=window_labels.to_numpy(), random_state=SEED
)
train_df = sample_rows(train_full, TRAIN_MAX_ROWS, SEED, TRAIN_ATTACK_FRACTION)
select_df = sample_rows(validation_full[validation_full.window_start.isin(select_windows)], SELECT_MAX_ROWS, SEED+1)
calibration_df = sample_rows(validation_full[validation_full.window_start.isin(calibration_windows)], CALIBRATION_MAX_ROWS, SEED+2)
# No model is evaluated on these test rows until section 12.
test_df = sample_rows(test_full, TEST_MAX_ROWS, SEED+3)
for name, df in [("train", train_df), ("selection", select_df), ("calibration", calibration_df), ("test", test_df)]:
    df[["edge_id"] + KEY_COLUMNS + ["split"]].to_parquet(OUTPUT_DIR / f"{name}_manifest.parquet", index=False)
display(pd.DataFrame({n: {"rows": len(d), "attacks": int(d.binary_target.sum())}
                      for n, d in [("train", train_df), ("selection", select_df), ("calibration", calibration_df), ("test", test_df)]}).T)


,rows,attacks
train,10000,5000
selection,1000,85
calibration,2000,164
test,2000,164


## 5. Label vocabulary and decision questions

Vocabulary comes only from the full training split. Held-out labels are never moved into training. The coverage table reports labels outside that vocabulary. If any exist, the multilabel score covers the training vocabulary only; report that limitation alongside the score.

Each label has its own two-option choice. Technique names below were checked against MITRE ATT&CK's Enterprise STIX dataset. Descriptions are short summaries for this experiment, not claims that a specific traffic pattern proves a technique.


In [6]:
TECHNIQUE_NAMES = {
    "T1557": "Adversary-in-the-Middle",
    "T1592": "Gather Victim Host Information",
    "T1133": "External Remote Services",
    "T1547": "Boot or Logon Autostart Execution",
    "T1071": "Application Layer Protocol",
    "T1190": "Exploit Public-Facing Application",
    "T1589": "Gather Victim Identity Information",
    "T1112": "Modify Registry",
    "T1595": "Active Scanning",
    "T1548": "Abuse Elevation Control Mechanism",
    "T1016": "System Network Configuration Discovery",
    "T1059": "Command and Scripting Interpreter",
    "T1204": "User Execution",
    "T1590": "Gather Victim Network Information",
    "T1210": "Exploitation of Remote Services",
    "T1048": "Exfiltration Over Alternative Protocol",
    "T1566": "Phishing",
    "T1110": "Brute Force",
    "T1078": "Valid Accounts",
    "T1571": "Non-Standard Port",
    "T1546": "Event Triggered Execution",
    "T1203": "Exploitation for Client Execution",
    "T1505": "Server Software Component",
    "T1136": "Create Account",
    "T1018": "Remote System Discovery",
    "T1046": "Network Service Discovery",
    "T1587": "Develop Capabilities"
}

TECHNIQUE_HINTS = {
    "T1046": "Discover services exposed by network hosts.",
    "T1595": "Actively probe a target to gather reconnaissance information.",
    "T1110": "Attempt repeated guesses or trials to obtain credentials.",
    "T1587": "Develop capabilities for use in later attacks.",
    "T1018": "Identify other systems on a network.",
    "T1592": "Gather information about victim hosts before compromise.",
    "T1590": "Gather information about a victim network before compromise.",
    "T1016": "Discover network configuration on a system.",
    "T1548": "Abuse mechanisms that control elevation of privileges.",
    "T1078": "Use legitimate account credentials for adversary access.",
    "T1190": "Exploit a vulnerability in an externally accessible application.",
    "T1048": "Transfer stolen data using an alternative network protocol.",
    "T1210": "Exploit remote services to access other systems.",
    "T1589": "Gather information about victim identities before compromise.",
    "T1203": "Exploit client software to execute malicious code.",
    "T1071": "Use application-layer protocols for command and control.",
    "T1566": "Send deceptive messages to obtain access or execution.",
    "T1204": "Rely on user actions to execute malicious content.",
    "T1059": "Execute commands through a command or scripting interpreter.",
    "T1547": "Configure automatic execution during boot or logon.",
    "T1136": "Create accounts to support adversary access.",
    "T1112": "Modify operating-system registry settings.",
    "T1571": "Use protocols on ports where they are not normally expected.",
    "T1557": "Position an adversary between communicating parties.",
    "T1546": "Trigger malicious execution through system or application events.",
    "T1505": "Abuse server software components for persistent access.",
    "T1133": "Use external remote services to gain or maintain access.",
}
TACTIC_HINTS = {
    "Reconnaissance": "Gather information to plan an attack.",
    "Resource Development": "Establish resources for an attack.",
    "Initial Access": "Gain an initial foothold.",
    "Execution": "Run adversary-controlled code.",
    "Persistence": "Maintain access across interruptions.",
    "Privilege Escalation": "Obtain higher permissions.",
    "Defense Evasion": "Avoid detection or security controls.",
    "Credential Access": "Obtain account credentials.",
    "Discovery": "Learn about an accessed environment.",
    "Lateral Movement": "Move to other systems.",
    "Collection": "Gather data of interest.",
    "Command and Control": "Communicate with compromised systems.",
    "Exfiltration": "Remove data from the environment.",
}

TACTICS = sorted({v for labels in train_full.tactic_targets for v in labels})
TECHNIQUES = sorted({v for labels in train_full.mitre_targets for v in labels})
coverage = []
for split, frame in [("validation", validation_full), ("test", test_full)]:
    for family, vocab in [("tactic", TACTICS), ("mitre", TECHNIQUES)]:
        unseen = sorted({x for labels in frame[family + "_targets"] for x in labels} - set(vocab))
        coverage.append({"split": split, "task": family, "labels_outside_train_vocabulary": unseen})
display(pd.DataFrame(coverage))

QUESTIONS = {"detection": {
    "type": "choice",
    "instructions": "Classify this five-minute source-to-destination communication using the traffic evidence.",
    "criteria": {"A": "Benign communication.", "B": "Attack-related communication."},
}}
QUESTION_META = [{"qid": "detection", "task": "binary", "label": "attack"}]
if TASK_MODE == "all":
    missing = set(TECHNIQUES) - set(TECHNIQUE_NAMES)
    if missing:
        raise ValueError(f"Add verified names and definitions for these training labels: {sorted(missing)}")
    for family, labels in [("tactic", TACTICS), ("mitre", TECHNIQUES)]:
        for label in labels:
            qid = family + "_" + label
            description = (label + ": " + TACTIC_HINTS.get(label, label)) if family == "tactic" else (
                label + " " + TECHNIQUE_NAMES[label] + ": " + TECHNIQUE_HINTS[label])
            QUESTIONS[qid] = {
                "type": "choice",
                "instructions": "Is this attack label present in the communication? " + description,
                "criteria": {"A": "Absent.", "B": "Present."},
            }
            QUESTION_META.append({"qid": qid, "task": family, "label": label})
QIDS = [m["qid"] for m in QUESTION_META]
TASK_COLUMNS = {task: [i for i, m in enumerate(QUESTION_META) if m["task"] == task]
                for task in ["binary", "tactic", "mitre"]}
(OUTPUT_DIR / "questions.json").write_text(json.dumps(QUESTIONS, indent=2))
(OUTPUT_DIR / "question_metadata.json").write_text(json.dumps(QUESTION_META, indent=2))
print("Training vocabulary:", len(TACTICS), "tactics;", len(TECHNIQUES), "techniques")
print("Active decisions per edge:", len(QIDS))


,split,task,labels_outside_train_vocabulary
0,validation,tactic,[]
1,validation,mitre,[]
2,test,tactic,[]
3,test,mitre,[]


Training vocabulary: 13 tactics; 27 techniques
Active decisions per edge: 1


## 6. Deterministic traffic text and masked targets

All numbers are generated from rows, with raw units and compact precision. Each summary includes only the feature whitelist. Labels remain separate. The original port field is marked as an ICMP type/code field when the row is exclusively ICMP, rather than calling it a transport port.


In [7]:
def fmt(value):
    return format(float(value), ".6g")

def state_from_row(r):
    port_name = "destination_port"
    if r["icmp_count"] > 0 and r["tcp_count"] == 0 and r["udp_count"] == 0:
        port_name = "ICMP_destination_field"
    parts = [
        "Five-minute communication from Host A to Host B.",
        f"connections={fmt(r['connection_count'])}; TCP={fmt(r['tcp_count'])}; UDP={fmt(r['udp_count'])}; ICMP={fmt(r['icmp_count'])}.",
        f"{port_name}={fmt(r['most_common_destination_port'])}; unique_destination_fields={fmt(r['unique_destination_ports'])}.",
        f"duration_seconds: total={fmt(r['total_duration'])}, mean={fmt(r['average_duration'])}, max={fmt(r['maximum_duration'])}.",
        f"payload_bytes: sent={fmt(r['total_orig_bytes'])}, response={fmt(r['total_resp_bytes'])}.",
        f"packets: sent={fmt(r['total_orig_packets'])}, response={fmt(r['total_resp_packets'])}.",
        f"IP_bytes: sent={fmt(r['total_orig_ip_bytes'])}, response={fmt(r['total_resp_ip_bytes'])}; missed_bytes={fmt(r['total_missed_bytes'])}.",
        f"service={r['most_common_service']}; Zeek_connection_state={r['most_common_connection_state']}.",
    ]
    if INCLUDE_NETWORK_CONTEXT:
        parts += [
            f"Host A in this window: destinations={fmt(r['src_unique_destinations'])}, outgoing_connections={fmt(r['src_connection_count'])}, sent_bytes={fmt(r['src_sent_bytes'])}.",
            f"Host B in this window: sources={fmt(r['dst_unique_sources'])}, incoming_connections={fmt(r['dst_connection_count'])}, response_bytes={fmt(r['dst_response_bytes'])}.",
        ]
    return " ".join(parts)

def target_arrays(frame):
    y = np.zeros((len(frame), len(QIDS)), dtype=np.int8)
    mask = np.ones_like(y, dtype=bool)
    for j, meta in enumerate(QUESTION_META):
        task, label = meta["task"], meta["label"]
        if task == "binary":
            y[:, j] = frame.binary_target.to_numpy()
        else:
            y[:, j] = frame[task + "_targets"].map(lambda values: label in values).to_numpy()
            mask[:, j] = frame["observed_" + task].to_numpy()
    return y, mask

def training_rows(frame):
    y, observed = target_arrays(frame)
    for i, row in enumerate(frame.to_dict("records")):
        yield {"state": state_from_row(row), "questions": QUESTIONS,
               "expected": {qid: "B" if y[i, j] else "A"
                            for j, qid in enumerate(QIDS) if observed[i, j]}}

with (OUTPUT_DIR / "training_cases.jsonl").open("w") as f:
    for row in training_rows(train_df):
        f.write(json.dumps(row) + "\n")
print(state_from_row(train_df.iloc[0]))
print("Expected labels are saved separately from the model input.")


Five-minute communication from Host A to Host B. connections=1; TCP=0; UDP=1; ICMP=0. destination_port=53; unique_destination_fields=1. duration_seconds: total=1.50204e-05, mean=1.50204e-05, max=1.50204e-05. payload_bytes: sent=76, response=0. packets: sent=2, response=0. IP_bytes: sent=132, response=0; missed_bytes=0. service=dns; Zeek_connection_state=S0. Host A in this window: destinations=1, outgoing_connections=1, sent_bytes=76. Host B in this window: sources=17, incoming_connections=32, response_bytes=0.
Expected labels are saved separately from the model input.


## 7. Load the base checkpoint and check token budgets

The root English checkpoint is used explicitly. Its resolved Hugging Face commit and input budgets are recorded. Baseline and fine-tuned models receive the same summaries and questions.

The check below examines every selected development/training summary against the smallest actual question-specific state budget. It stops on truncation. Shorten the template, disable network context for both models, or deliberately increase MAX_LEN for both models and restart with a new run name. Do not silently discard input features.


In [8]:
BASE_PATH = snapshot_download(
    MODEL_ID, revision=MODEL_REVISION,
    allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"],
)
(OUTPUT_DIR / "base_checkpoint.json").write_text(json.dumps({
    "model_id": MODEL_ID, "requested_revision": MODEL_REVISION,
    "resolved_commit": Path(BASE_PATH).name, "local_snapshot": BASE_PATH,
}, indent=2))
model, tokenizer, base_cfg = load_checkpoint(BASE_PATH)
if base_cfg.get("option_layout", "sequential") != "sequential":
    raise ValueError("This notebook expects the root checkpoint's sequential option layout.")
base_cfg = copy.deepcopy(base_cfg)
base_cfg.update(max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
model.to(DEVICE).eval()

INTERNAL_QUESTIONS = {qid: to_internal(qid, QUESTIONS[qid]) for qid in QIDS}
rooms = {}
for qid, q in INTERNAL_QUESTIONS.items():
    head, markers, stats = build_head(tokenizer, q, HEAD_MAX_LEN)
    if len(markers) != 2 or stats["options_distinct"] != 2:
        raise ValueError(f"Question options collapse: {qid}")
    if stats["tokens_per_option"] is not None:
        raise ValueError(f"Option descriptions are being shortened: {qid}")
    rooms[qid] = MAX_LEN - len(head) - 1
MIN_STATE_ROOM = min(rooms.values())

def check_text_budget(frame, name):
    lengths = []
    for start in tqdm(range(0, len(frame), 256), desc=f"Token check: {name}"):
        texts = [state_from_row(r) for r in frame.iloc[start:start+256].to_dict("records")]
        ids = tokenizer(texts, add_special_tokens=False, truncation=False)["input_ids"]
        lengths.extend(map(len, ids))
    if max(lengths, default=0) > MIN_STATE_ROOM:
        raise ValueError(f"{name}: {sum(n > MIN_STATE_ROOM for n in lengths)} rows exceed {MIN_STATE_ROOM} state tokens. Fix the template/budget for both models.")
    return {"subset": name, "max_state_tokens": max(lengths, default=0), "available": MIN_STATE_ROOM}

display(pd.DataFrame([check_text_budget(d, n) for n, d in [
    ("train", train_df), ("selection", select_df), ("calibration", calibration_df)
]]))


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Token check: train:   0%|          | 0/40 [00:00<?, ?it/s]

Token check: selection:   0%|          | 0/4 [00:00<?, ?it/s]

Token check: calibration:   0%|          | 0/8 [00:00<?, ?it/s]

,subset,max_state_tokens,available
0,train,207,473
1,selection,195,473
2,calibration,197,473


## 8. Batched prediction and metrics

The official Laya training adapter returns logits with the same question rendering used by its SDK. In prediction, dummy answers only satisfy the adapter's record schema. They are never fed into the model; only the state and question tokens are model inputs. Predictions are made for every active question, even when its ground truth is missing. Missing annotations affect scoring only.

Scores use the probability of the positive option, not the model's confidence field. Average precision is reported as PR-AUC (AP). Multilabel macro-F1 is reported over all training labels and over labels with positive support. Unknown annotation rows are excluded by task.


In [9]:
def predict_margins(model, tok, frame):
    model.eval()
    result = np.empty((len(frame), len(QIDS)), dtype=np.float32)
    started = time.perf_counter()
    edge_batch = max(1, 64 // len(QIDS))
    for start in tqdm(range(0, len(frame), edge_batch), desc="Predict"):
        part = frame.iloc[start:start+edge_batch]
        rows = [{"state": state_from_row(r), "questions": QUESTIONS,
                 "expected": {qid: "A" for qid in QIDS}}
                for r in part.to_dict("records")]
        items, skipped = items_from_rows(tok, rows, MAX_LEN, HEAD_MAX_LEN)
        if skipped or len(items) != len(part) * len(QIDS):
            raise RuntimeError(f"Prediction item mismatch: {skipped}")
        records = calibration_records(model, tok, items, DEVICE, MAX_LEN, HEAD_MAX_LEN,
                                      batch_size=EVAL_BATCH_SIZE)
        # Criteria are always ordered A=negative, B=positive.
        margins = [float(logits[1] - logits[0]) for _, logits, _, k in records if k == 2]
        result[start:start+len(part)] = np.asarray(margins).reshape(len(part), len(QIDS))
    if not np.isfinite(result).all():
        raise FloatingPointError("Nonfinite prediction logits.")
    elapsed = time.perf_counter() - started
    return result, {"seconds": elapsed, "seconds_per_edge": elapsed/max(1, len(frame)),
                    "decisions_per_edge": len(QIDS), "rows": len(frame)}

def checkpoint_temperature(cfg):
    qt = QTYPES["choice"]
    return clamp_temperature(cfg.get("temperature_by_options", {}).get(
        temp_bucket(qt, 2), cfg.get("temperature", [1., 1., 1.])[qt]))

def fit_temperature(margins, frame):
    y, observed = target_arrays(frame)
    z, t = margins[observed].astype(float), y[observed].astype(float)
    if len(np.unique(t)) < 2:
        warnings.warn("Calibration labels have one class. Temperature remains 1.")
        return 1.0
    fit = minimize_scalar(lambda temperature: np.mean(np.logaddexp(0, z/temperature) - t*z/temperature),
                          bounds=(0.5, 5.0), method="bounded")
    if not fit.success:
        raise RuntimeError("Temperature fitting failed.")
    return float(fit.x)

def threshold_for_f1(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    score = 2*precision[:-1]*recall[:-1] / np.maximum(precision[:-1]+recall[:-1], 1e-12)
    best = np.flatnonzero(np.isclose(score, score.max()))
    return float(thresholds[best[np.argmin(np.abs(thresholds[best] - 0.5))]])

def tune_thresholds(frame, probabilities):
    y, observed = target_arrays(frame)
    thresholds = np.full(len(QIDS), 0.5)
    details = []
    for j, qid in enumerate(QIDS):
        t, p = y[observed[:, j], j], probabilities[observed[:, j], j]
        positives, negatives = int(t.sum()), int(len(t)-t.sum())
        enough = positives >= MIN_THRESHOLD_POSITIVES and negatives >= MIN_THRESHOLD_NEGATIVES
        if enough:
            thresholds[j] = threshold_for_f1(t, p)
        details.append({"qid": qid, "positives": positives, "negatives": negatives,
                        "threshold": thresholds[j], "tuned": enough})
    return thresholds, pd.DataFrame(details)

def metrics_tables(frame, probabilities, thresholds=0.5):
    y, observed = target_arrays(frame)
    pred = probabilities >= np.asarray(thresholds)
    summaries, per_label = [], []
    for task, js in TASK_COLUMNS.items():
        if not js:
            continue
        keep = observed[:, js].all(axis=1)
        yt, yp, pp = y[keep][:, js], pred[keep][:, js], probabilities[keep][:, js]
        positive_support = yt.sum(axis=0)
        if len(yt) == 0:
            summaries.append({"task": task, "evaluated_rows": 0})
            continue
        if task == "binary":
            t, p, hard = yt.ravel(), pp.ravel(), yp.ravel()
            tn, fp, fn, tp = confusion_matrix(t, hard, labels=[0, 1]).ravel()
            summaries.append({"task": task, "evaluated_rows": len(t), "positive_labels": int(t.sum()),
                "accuracy": accuracy_score(t, hard), "precision": precision_score(t, hard, zero_division=0),
                "recall": recall_score(t, hard, zero_division=0), "f1": f1_score(t, hard, zero_division=0),
                "pr_auc_ap": average_precision_score(t, p) if len(np.unique(t)) == 2 else np.nan,
                "roc_auc": roc_auc_score(t, p) if len(np.unique(t)) == 2 else np.nan,
                "brier": brier_score_loss(t, p), "tn": tn, "fp": fp, "fn": fn, "tp": tp})
        else:
            support = positive_support > 0
            summaries.append({"task": task, "evaluated_rows": len(yt), "positive_labels": int(yt.sum()),
                "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
                "macro_f1_all_labels": f1_score(yt, yp, average="macro", zero_division=0),
                "macro_f1_supported": float(np.mean([f1_score(yt[:, k], yp[:, k], zero_division=0)
                                                      for k in np.flatnonzero(support)])) if support.any() else np.nan,
                "supported_labels": int(support.sum()), "total_labels": len(js),
                "hamming_loss": float(np.mean(yt != yp)), "exact_match": float(np.mean((yt == yp).all(axis=1)))})
        for k, j in enumerate(js):
            t, p, hard = yt[:, k], pp[:, k], yp[:, k]
            per_label.append({**QUESTION_META[j], "evaluated_rows": len(t), "positive_support": int(t.sum()),
                "predicted_positive": int(hard.sum()), "precision": precision_score(t, hard, zero_division=0),
                "recall": recall_score(t, hard, zero_division=0), "f1": f1_score(t, hard, zero_division=0),
                "pr_auc_ap": average_precision_score(t, p) if len(np.unique(t)) == 2 else np.nan})
    return pd.DataFrame(summaries), pd.DataFrame(per_label)

def selection_score(frame, margins):
    # Equal weight per active task family, independent of label count.
    y, observed = target_arrays(frame)
    family_scores = []
    for task, js in TASK_COLUMNS.items():
        scores = []
        for j in js:
            t = y[observed[:, j], j]
            if len(np.unique(t)) == 2:
                scores.append(average_precision_score(t, margins[observed[:, j], j]))
        if scores:
            family_scores.append(float(np.mean(scores)))
    if not family_scores:
        raise ValueError("No selection labels have both classes. Increase SELECT_MAX_ROWS.")
    return float(np.mean(family_scores))


## 9. Zero-shot baseline on validation

The native baseline uses the shipped checkpoint's probability temperature and threshold 0.5. A separate validation-tuned threshold is saved for a fair comparison with the fine-tuned model. This second variant uses validation labels and must be described as **zero-shot weights with validation-tuned thresholds**, not a completely label-free baseline.


In [10]:
base_temperature = checkpoint_temperature(base_cfg)
baseline_cal_margins, baseline_cal_time = predict_margins(model, tokenizer, calibration_df)
baseline_cal_prob = expit(baseline_cal_margins / base_temperature)
baseline_thresholds, baseline_threshold_table = tune_thresholds(calibration_df, baseline_cal_prob)
baseline_threshold_table.to_csv(OUTPUT_DIR / "baseline_thresholds.csv", index=False)
baseline_validation_metrics, baseline_validation_labels = metrics_tables(calibration_df, baseline_cal_prob)
display(baseline_validation_metrics)
baseline_validation_metrics.to_csv(OUTPUT_DIR / "baseline_validation_metrics.csv", index=False)
baseline_validation_labels.to_csv(OUTPUT_DIR / "baseline_validation_per_label.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "baseline_calibration_predictions.npz",
                    edge_id=calibration_df.edge_id.to_numpy(dtype=str), margins=baseline_cal_margins,
                    probabilities=baseline_cal_prob)
print("These are development results, not final test results.")


Predict:   0%|          | 0/32 [00:00<?, ?it/s]

,task,evaluated_rows,positive_labels,accuracy,precision,recall,f1,pr_auc_ap,roc_auc,brier,tn,fp,fn,tp
0,binary,2000,164,0.6605,0.151556,0.682927,0.248062,0.310261,0.740415,0.234754,1209,627,52,112


These are development results, not final test results.


## 10. Fine-tune using observed training labels only

The training function below is Laya's official train_model API, using supervised cross-entropy by default. In all-task mode, negative auxiliary decisions are capped relative to positives per question; positive labels are retained. Questions without positive examples in the sampled training subset are excluded from auxiliary training and reported. Increase the training subset if coverage is insufficient.

Checkpoint selection uses validation AP for binary mode. All-task mode uses the mean of binary AP, tactic macro-AP, and technique macro-AP over selection labels with both classes. This avoids choosing a model only for the easiest task. Training uses four fixed epochs; the best checkpoint is retained.

For CUDA out-of-memory errors, reduce MICRO_BATCH and EVAL_BATCH_SIZE and increase GRAD_ACCUM to retain the effective training batch. Restart the experiment afterward. FREEZE_ENCODER=True is a lower-memory alternative and should be reported as head-only training.


In [11]:
if DEVICE.type != "cuda" and not ALLOW_CPU_TRAINING:
    raise RuntimeError("Use a CUDA GPU for this pilot, or explicitly enable ALLOW_CPU_TRAINING in section 2.")

train_items, skipped = items_from_rows(tokenizer, training_rows(train_df), MAX_LEN, HEAD_MAX_LEN)
if skipped:
    raise ValueError(f"Some training decisions could not be encoded: {skipped}")
# The official adapter omits question IDs. Restore them in its row/question iteration order.
_, train_observed = target_arrays(train_df)
item_qids = [qid for row_mask in train_observed for j, qid in enumerate(QIDS) if row_mask[j]]
assert len(item_qids) == len(train_items)
for item, qid in zip(train_items, item_qids):
    item["_qid"] = qid

def balance_training_items(items):
    rng = np.random.default_rng(SEED)
    by_question = {}
    for item in items:
        by_question.setdefault(item["_qid"], []).append(item)
    kept, report = [], []
    for qid in QIDS:
        group = by_question.get(qid, [])
        positive = [it for it in group if it["target"][1] > 0.5]
        negative = [it for it in group if it["target"][0] > 0.5]
        if qid != "detection" and AUX_NEGATIVES_PER_POSITIVE is not None:
            n = min(len(negative), int(AUX_NEGATIVES_PER_POSITIVE * len(positive)))
            indices = rng.choice(len(negative), size=n, replace=False) if n else []
            negative = [negative[i] for i in indices]
        selected = positive + negative
        kept.extend(selected)
        report.append({"qid": qid, "positive_decisions": len(positive), "negative_decisions": len(negative),
                       "total": len(selected), "has_positive_training_examples": bool(positive)})
    return kept, pd.DataFrame(report)

train_items, training_support = balance_training_items(train_items)
display(training_support)
training_support.to_csv(OUTPUT_DIR / "training_decision_support.csv", index=False)
if not training_support.has_positive_training_examples.all():
    warnings.warn("Some labels have no positive training examples in this pilot. See training_decision_support.csv.")

training_cfg = TrainConfig(
    epochs=EPOCHS, micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM,
    encoder_lr=ENCODER_LR, head_lr=HEAD_LR, loss=LOSS,
    shuffle_options=("choice",), freeze_encoder=FREEZE_ENCODER,
    seed=SEED, max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN,
    amp=(DEVICE.type == "cuda"), gradient_checkpointing=(DEVICE.type == "cuda"),
    log_every=100,
)
print("Training decisions:", len(train_items))
print("Approximate optimizer updates:", EPOCHS * int(np.ceil(np.ceil(len(train_items)/MICRO_BATCH)/GRAD_ACCUM)))

BEST_DIR = OUTPUT_DIR / "best_checkpoint"
history = []
best = {"score": -np.inf, "epoch": None}
fine_cfg = copy.deepcopy(base_cfg)
fine_cfg["temperature"] = [1., 1., 1.]
fine_cfg.pop("temperature_by_options", None)
fine_cfg.pop("binning_map", None)
fine_cfg["training"] = {"experiment": RUN_NAME, "laya_train": asdict(training_cfg)}

def on_epoch_end(epoch, mean_loss):
    try:
        margins, timing = predict_margins(model, tokenizer, select_df)
        score = selection_score(select_df, margins)
        history.append({"epoch": epoch+1, "train_loss": mean_loss, "selection_ap": score})
        if score > best["score"]:
            best.update(score=score, epoch=epoch+1)
            save_checkpoint(model, tokenizer, fine_cfg, str(BEST_DIR))
        pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
        (OUTPUT_DIR / "best_selection.json").write_text(json.dumps(best, indent=2))
        print(f"Epoch {epoch+1}: selection AP={score:.4f}; best epoch={best['epoch']}")
    finally:
        # Laya's callback runs inside its training loop. Restore training mode after evaluation.
        model.train()
        if FREEZE_ENCODER:
            model.encoder.eval()

train_losses = train_model(model, tokenizer, train_items, training_cfg, DEVICE,
                          MAX_LEN, HEAD_MAX_LEN, on_epoch_end=on_epoch_end)
del train_items
print("Selected checkpoint:", BEST_DIR, "|", best)


,qid,positive_decisions,negative_decisions,total,has_positive_training_examples
0,detection,5000,5000,10000,True


Training decisions: 10000
Approximate optimizer updates: 628



KeyboardInterrupt



## 11. Calibrate the selected model and freeze thresholds

Only the calibration half of validation is used here. A single temperature is fitted across observed choice decisions, matching the checkpoint's two-option choice bucket. Per-label thresholds maximize calibration F1 when at least 10 positive and 10 negative examples are available; otherwise they remain 0.5. These choices are fixed before any test prediction.

Calibration and threshold fitting share this validation subset, so its tuned metrics are development estimates. Final generalization is measured on the test split.


In [ ]:
del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
model, tokenizer, selected_cfg = load_checkpoint(str(BEST_DIR))
model.to(DEVICE).eval()
fine_cal_margins, fine_cal_time = predict_margins(model, tokenizer, calibration_df)
fine_temperature = fit_temperature(fine_cal_margins, calibration_df)
fine_cal_prob = expit(fine_cal_margins / fine_temperature)
fine_thresholds, fine_threshold_table = tune_thresholds(calibration_df, fine_cal_prob)
fine_threshold_table.to_csv(OUTPUT_DIR / "finetuned_thresholds.csv", index=False)
fine_validation_metrics, fine_validation_labels = metrics_tables(calibration_df, fine_cal_prob, fine_thresholds)
fine_validation_metrics.to_csv(OUTPUT_DIR / "finetuned_validation_metrics.csv", index=False)
fine_validation_labels.to_csv(OUTPUT_DIR / "finetuned_validation_per_label.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "finetuned_calibration_predictions.npz",
                    edge_id=calibration_df.edge_id.to_numpy(dtype=str), margins=fine_cal_margins,
                    probabilities=fine_cal_prob)

FINAL_DIR = OUTPUT_DIR / "finetuned_checkpoint"
selected_cfg["temperature"] = [1., 1., 1.]
selected_cfg["temperature"][QTYPES["choice"]] = fine_temperature
selected_cfg["temperature_by_options"] = {temp_bucket(QTYPES["choice"], 2): fine_temperature}
save_checkpoint(model, tokenizer, selected_cfg, str(FINAL_DIR))
decision_settings = {
    "question_ids": QIDS, "base_temperature": base_temperature, "fine_temperature": fine_temperature,
    "baseline_thresholds": baseline_thresholds.tolist(), "finetuned_thresholds": fine_thresholds.tolist(),
    "calibration_source": "validation calibration windows only", "selected_epoch": best["epoch"],
}
(OUTPUT_DIR / "decision_settings.json").write_text(json.dumps(decision_settings, indent=2))
print("Fine-tuned temperature:", round(fine_temperature, 4))
display(fine_validation_metrics)


## 12. Final test comparison

Run after the settings above are fixed. Both models use exactly the same test edge IDs. The test subset keeps approximately the original class prevalence. If TEST_MAX_ROWS is set, label the results as a pilot, not the full test benchmark.

Four variants are reported: native zero-shot at 0.5, zero-shot weights with validation-tuned thresholds, fine-tuned/calibrated at 0.5, and fine-tuned/calibrated with validation-tuned thresholds. Also report an always-benign reference to expose misleading accuracy on this imbalanced dataset.

Do not paste earlier GraphSAGE scores into this comparison: its original training/evaluation did not mask missing attack annotations. Re-evaluate GraphSAGE on the same IDs and task masks for a fair comparison.


In [ ]:
TEST_RESULTS_PATH = OUTPUT_DIR / "test_comparison.csv"
if TEST_RESULTS_PATH.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError("Final test results already exist. Read the saved comparison instead of rerunning test-driven experiments.")
check_text_budget(test_df, "test")

# Reload the exact saved checkpoint so reported scores correspond to the deployable file.
del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
model, tokenizer, _ = load_checkpoint(str(FINAL_DIR))
model.to(DEVICE).eval()
fine_test_margins, fine_test_time = predict_margins(model, tokenizer, test_df)
fine_test_prob = expit(fine_test_margins / fine_temperature)
del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

model, tokenizer, _ = load_checkpoint(BASE_PATH)
model.to(DEVICE).eval()
base_test_margins, base_test_time = predict_margins(model, tokenizer, test_df)
base_test_prob = expit(base_test_margins / base_temperature)
del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

comparisons = []
for name, probabilities, thresholds in [
    ("zero_shot_native_0.5", base_test_prob, 0.5),
    ("zero_shot_validation_thresholds", base_test_prob, baseline_thresholds),
    ("finetuned_calibrated_0.5", fine_test_prob, 0.5),
    ("finetuned_validation_thresholds", fine_test_prob, fine_thresholds),
    ("always_benign", np.zeros_like(base_test_prob), 0.5),
]:
    summary, details = metrics_tables(test_df, probabilities, thresholds)
    summary.insert(0, "variant", name)
    comparisons.append(summary)
    details.to_csv(OUTPUT_DIR / f"test_per_label_{name}.csv", index=False)
comparison = pd.concat(comparisons, ignore_index=True)
comparison.to_csv(TEST_RESULTS_PATH, index=False)
np.savez_compressed(OUTPUT_DIR / "test_predictions.npz",
                    edge_id=test_df.edge_id.to_numpy(dtype=str), question_ids=np.asarray(QIDS),
                    truth=target_arrays(test_df)[0], observed=target_arrays(test_df)[1],
                    baseline=base_test_prob, finetuned=fine_test_prob)
(OUTPUT_DIR / "inference_timing.json").write_text(json.dumps({
    "baseline": base_test_time, "finetuned": fine_test_time,
    "note": "Batched fp32 notebook pipeline including text encoding; excludes model loading. Not a production latency benchmark."
}, indent=2))
display(comparison)


## 13. Inspect training and per-label performance

Check binary false positives and recall, plus tactic/technique macro-F1 and support. Large gains in micro-F1 can hide failures on rare labels. Repeat the chosen development configuration with several seeds before making a strong research claim. Keep the test protocol fixed.


In [ ]:
history_df = pd.read_csv(OUTPUT_DIR / "training_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history_df.epoch, history_df.train_loss, marker="o")
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy / selected loss")
axes[1].plot(history_df.epoch, history_df.selection_ap, marker="o")
axes[1].set(title="Validation checkpoint score", xlabel="Epoch", ylabel="Selection AP")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=160)
plt.show()
display(pd.read_csv(OUTPUT_DIR / "test_per_label_finetuned_validation_thresholds.csv"))
print("Saved results and models:", OUTPUT_DIR.resolve())


## 14. Predict new rows with the saved model

This optional cell shows deployment through Laya's public SDK. Supply a row in raw units with the same feature columns. If network context is enabled, compute it from that row's complete five-minute window first. No true labels are required by this function. Use the saved question schema and thresholds unchanged.


In [ ]:
# Run this cell when ready to predict new data; it loads another model into memory.
# import laya
# agent = laya.load(str(FINAL_DIR), device=str(DEVICE))
# saved_questions = json.loads((OUTPUT_DIR / "questions.json").read_text())
# saved_settings = json.loads((OUTPUT_DIR / "decision_settings.json").read_text())
#
# def predict_new_row(raw_row):
#     result = agent.predict(state_from_row(raw_row), saved_questions)
#     if result.get("usage", {}).get("truncated"):
#         raise ValueError("New input was truncated; review the feature template.")
#     scores = {qid: float(result["answers"][qid]["probabilities"]["B"])
#               for qid in saved_settings["question_ids"]}
#     labels = {qid: int(scores[qid] >= threshold)
#               for qid, threshold in zip(saved_settings["question_ids"], saved_settings["finetuned_thresholds"])}
#     return {"probabilities": scores, "predictions": labels}
